# ML-08 — Capstone Modeling Lane

## 1. Method choice and why

**Method:** Random Forest Classification

I selected Random Forest because the dataset contains a mix of numerical and categorical features and the target (`trend_direction`) is categorical. Random Forest can capture non-linear relationships, is robust to noisy features, and provides feature importance for interpretation.

The goal is to compare this model against a simple Week-4 style baseline on the same split and metric.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv('content_refresh_anonymized(1).csv')

target = 'trend_direction'

X = df.drop(columns=[target, 'content_id', 'client_id'])
y = df[target]

X.head()


## 2. Split design

A stratified train/test split is used so that the class distribution of `trend_direction` is preserved in both train and test sets.

This is an honest evaluation because the baseline and model are evaluated on the exact same test data.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

categorical = X.select_dtypes(include='object').columns.tolist()

preprocessor = ColumnTransformer(
    [('cat', OneHotEncoder(handle_unknown='ignore'), categorical)],
    remainder='passthrough'
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)


## 3. Train + compare vs my baseline

Baseline = most frequent class.

Model = Random Forest.

Metric = Accuracy.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

baseline = Pipeline([
    ('prep', preprocessor),
    ('model', DummyClassifier(strategy='most_frequent'))
])

rf = Pipeline([
    ('prep', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

baseline.fit(X_train, y_train)
rf.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)
rf_pred = rf.predict(X_test)

baseline_acc = accuracy_score(y_test, baseline_pred)
rf_acc = accuracy_score(y_test, rf_pred)

comparison = pd.DataFrame({
    'Model':['Baseline','Random Forest'],
    'Accuracy':[baseline_acc, rf_acc]
})

comparison


## 4. Errors and interpretation

Review the confusion matrix and classification report. Identify where the model makes mistakes and whether certain classes are harder to predict.

Also inspect feature importance from the Random Forest to understand which variables contribute most strongly to predictions.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, rf_pred))
print(confusion_matrix(y_test, rf_pred))

print("\nBaseline Accuracy:", round(baseline_acc,4))
print("Random Forest Accuracy:", round(rf_acc,4))


## Self-check

- [x] Method choice explained
- [x] Split design explained
- [x] Baseline comparison included
- [x] Same metric used
- [x] Error analysis section included
- [x] Notebook ready to execute top-to-bottom
